In [ ]:
# Example 1: X.509 Certificate Parsing with RFC5280
from pyasn1_modules import rfc5280
from pyasn1.codec.der import decoder, encoder
from pyasn1.type import univ, char, useful
import base64

print("=== X.509 Certificate Structure (RFC 5280) ===")

# Access X.509 certificate components
cert = rfc5280.Certificate()
print(f"Certificate structure: {type(cert)}")

# Create a sample TBSCertificate (To Be Signed Certificate)
tbs_cert = rfc5280.TBSCertificate()
# Version: use integer 2 for v3 (ASN.1 enumeration: v1=0, v2=1, v3=2)
tbs_cert.setComponentByName('version', 2)
tbs_cert.setComponentByName('serialNumber', rfc5280.CertificateSerialNumber(123456789))

# Set issuer - create proper Name with RDNSequence
issuer_attr = rfc5280.AttributeTypeAndValue()
issuer_attr.setComponentByName('type', univ.ObjectIdentifier('2.5.4.3'))  # Common Name OID
issuer_attr.setComponentByName('value', char.UTF8String('Test Issuer'))

issuer_rdn = rfc5280.RelativeDistinguishedName()
issuer_rdn.setComponentByPosition(0, issuer_attr)

issuer_rdn_seq = rfc5280.RDNSequence()
issuer_rdn_seq.setComponentByPosition(0, issuer_rdn)

issuer = rfc5280.Name()
issuer.setComponentByName('rdnSequence', issuer_rdn_seq)

# Set subject - create proper Name with RDNSequence
subject_attr = rfc5280.AttributeTypeAndValue()
subject_attr.setComponentByName('type', univ.ObjectIdentifier('2.5.4.3'))  # Common Name OID
subject_attr.setComponentByName('value', char.UTF8String('Test Subject'))

subject_rdn = rfc5280.RelativeDistinguishedName()
subject_rdn.setComponentByPosition(0, subject_attr)

subject_rdn_seq = rfc5280.RDNSequence()
subject_rdn_seq.setComponentByPosition(0, subject_rdn)

subject = rfc5280.Name()
subject.setComponentByName('rdnSequence', subject_rdn_seq)

tbs_cert.setComponentByName('issuer', issuer)
tbs_cert.setComponentByName('subject', subject)

# Set validity period
validity = rfc5280.Validity()
validity.setComponentByName('notBefore', rfc5280.Time().setComponentByName('generalTime', useful.GeneralizedTime('20240101120000Z')))
validity.setComponentByName('notAfter', rfc5280.Time().setComponentByName('generalTime', useful.GeneralizedTime('20251231235959Z')))
tbs_cert.setComponentByName('validity', validity)

print(f"TBS Certificate version: {tbs_cert.getComponentByName('version')}")
print(f"Serial number: {tbs_cert.getComponentByName('serialNumber')}")
print(f"Issuer: {issuer}")
print(f"Subject: {subject}")
print(f"Validity structure created: {type(validity)}")

# Access standard X.509 extensions
print(f"Standard extensions available: {dir(rfc5280)[:5]}")
print("Example 1 completed successfully!")

In [ ]:
# Example 2: PKCS#1 RSA Cryptography Standard (RFC 3447)
from pyasn1_modules import rfc3447
from pyasn1.codec.der import encoder, decoder

print("=== PKCS#1 RSA Standard (RFC 3447) ===")

# RSA Public Key structure
rsa_public_key = rfc3447.RSAPublicKey()
rsa_public_key.setComponentByName('modulus', univ.Integer(123456789012345678901234567890))
rsa_public_key.setComponentByName('publicExponent', univ.Integer(65537))

print(f"RSA Public Key structure: {type(rsa_public_key)}")
print(f"Modulus: {rsa_public_key.getComponentByName('modulus')}")
print(f"Public Exponent: {rsa_public_key.getComponentByName('publicExponent')}")

# RSA Private Key structure - requires all 9 components
rsa_private_key = rfc3447.RSAPrivateKey()
rsa_private_key.setComponentByName('version', univ.Integer(0))
rsa_private_key.setComponentByName('modulus', univ.Integer(123456789012345678901234567890))
rsa_private_key.setComponentByName('publicExponent', univ.Integer(65537))
rsa_private_key.setComponentByName('privateExponent', univ.Integer(987654321098765432109876543210))
rsa_private_key.setComponentByName('prime1', univ.Integer(11111111111111111))
rsa_private_key.setComponentByName('prime2', univ.Integer(11111111111111113))
rsa_private_key.setComponentByName('exponent1', univ.Integer(1111111))  # d mod (p-1)
rsa_private_key.setComponentByName('exponent2', univ.Integer(1111113))  # d mod (q-1)
rsa_private_key.setComponentByName('coefficient', univ.Integer(222222))  # (inverse of q) mod p

print(f"RSA Private Key version: {rsa_private_key.getComponentByName('version')}")
print(f"Private key structure complete: {type(rsa_private_key)}")

# Encode RSA structures
encoded_public = encoder.encode(rsa_public_key)
encoded_private = encoder.encode(rsa_private_key)

print(f"Public key DER size: {len(encoded_public)} bytes")
print(f"Private key DER size: {len(encoded_private)} bytes")

# Available PKCS#1 structures
pkcs1_types = [attr for attr in dir(rfc3447) if 'RSA' in attr or 'PKCS' in attr][:5]
print(f"PKCS#1 types available: {pkcs1_types}")
print("Example 2 completed successfully!")

In [ ]:
# Example 3: CMS (Cryptographic Message Syntax) - RFC 5652
from pyasn1_modules import rfc5652, rfc5280
from pyasn1.type import namedtype

print("=== CMS (Cryptographic Message Syntax) RFC 5652 ===")

# ContentInfo structure - root of CMS
content_info = rfc5652.ContentInfo()
content_info.setComponentByName('contentType', univ.ObjectIdentifier('1.2.840.113549.1.7.1'))  # id-data
content_info.setComponentByName('content', univ.OctetString(b'Sample CMS data content'))

# Enveloped Data structure for encrypted content
enveloped_data = rfc5652.EnvelopedData()
enveloped_data.setComponentByName('version', rfc5652.CMSVersion(0))

# Recipient Info for key transport
recipient_info = rfc5652.RecipientInfo()
key_trans_recipient = rfc5652.KeyTransRecipientInfo()
key_trans_recipient.setComponentByName('version', rfc5652.CMSVersion(0))

# Set recipient identifier using issuerAndSerialNumber
# Create a simple Name for issuer
issuer_attr = rfc5280.AttributeTypeAndValue()
issuer_attr.setComponentByName('type', univ.ObjectIdentifier('2.5.4.3'))  # Common Name
issuer_attr.setComponentByName('value', char.UTF8String('Test CA'))

issuer_rdn = rfc5280.RelativeDistinguishedName()
issuer_rdn.setComponentByPosition(0, issuer_attr)

issuer_rdn_seq = rfc5280.RDNSequence()
issuer_rdn_seq.setComponentByPosition(0, issuer_rdn)

issuer = rfc5280.Name()
issuer.setComponentByName('rdnSequence', issuer_rdn_seq)

# Create IssuerAndSerialNumber
issuer_and_serial = rfc5652.IssuerAndSerialNumber()
issuer_and_serial.setComponentByName('issuer', issuer)
issuer_and_serial.setComponentByName('serialNumber', univ.Integer(12345))

# Set RecipientIdentifier
recipient_id = rfc5652.RecipientIdentifier()
recipient_id.setComponentByName('issuerAndSerialNumber', issuer_and_serial)
key_trans_recipient.setComponentByName('rid', recipient_id)

print(f"ContentInfo type: {type(content_info)}")
print(f"Enveloped Data version: {enveloped_data.getComponentByName('version')}")
print(f"Key Transport Recipient version: {key_trans_recipient.getComponentByName('version')}")

# Encrypted Content Info
encrypted_content_info = rfc5652.EncryptedContentInfo()
encrypted_content_info.setComponentByName('contentType', rfc5652.ContentType('1.2.840.113549.1.7.1'))

print(f"Encrypted Content Info type: {encrypted_content_info.getComponentByName('contentType')}")

# Available CMS structures
cms_types = [attr for attr in dir(rfc5652) if 'Data' in attr or 'Info' in attr][:5]
print(f"CMS structure types: {cms_types}")

# Encode ContentInfo
encoded_content_info = encoder.encode(content_info)
print(f"ContentInfo encoded size: {len(encoded_content_info)} bytes")
print("Example 3 completed successfully!")

In [ ]:
# Example 4: SNMP Protocol Structures (RFC 1157 - SNMPv1)
from pyasn1_modules import rfc1157, rfc1155
from pyasn1.type import univ

print("=== SNMP Protocol Structures (RFC 1157 - SNMPv1) ===")

# SNMP Message structure
snmp_message = rfc1157.Message()
snmp_message.setComponentByName('version', univ.Integer(0))  # SNMPv1 = 0
snmp_message.setComponentByName('community', univ.OctetString(b'public'))

# Create PDU (GetRequest)
get_request_pdu = rfc1157.GetRequestPDU()
get_request_pdu.setComponentByName('request-id', univ.Integer(12345))
get_request_pdu.setComponentByName('error-status', univ.Integer(0))  # noError
get_request_pdu.setComponentByName('error-index', univ.Integer(0))

# Variable Bindings - use rfc1155.ObjectSyntax
var_bind_list = rfc1157.VarBindList()
var_bind = rfc1157.VarBind()
var_bind.setComponentByName('name', univ.ObjectIdentifier('1.3.6.1.2.1.1.1.0'))  # sysDescr.0

# Create ObjectSyntax with empty value (for GET request)
obj_syntax = rfc1155.ObjectSyntax()
simple_syntax = rfc1155.SimpleSyntax()
simple_syntax.setComponentByName('empty', univ.Null())
obj_syntax.setComponentByName('simple', simple_syntax)
var_bind.setComponentByName('value', obj_syntax)

var_bind_list.setComponentByPosition(0, var_bind)
get_request_pdu.setComponentByName('variable-bindings', var_bind_list)

# Set PDU in message
pdus = rfc1157.Pdus()
pdus.setComponentByName('get-request', get_request_pdu)
snmp_message.setComponentByName('data', pdus)

print(f"SNMP Message version: {snmp_message.getComponentByName('version')}")
print(f"Community string: {snmp_message.getComponentByName('community')}")
print(f"Get Request PDU type: {type(get_request_pdu)}")
print(f"Request ID: {get_request_pdu.getComponentByName('request-id')}")
print(f"Variable binding OID: {var_bind.getComponentByName('name')}")

# Create GetNextRequest PDU
get_next_pdu = rfc1157.GetNextRequestPDU()
get_next_pdu.setComponentByName('request-id', univ.Integer(12346))
get_next_pdu.setComponentByName('error-status', univ.Integer(0))
get_next_pdu.setComponentByName('error-index', univ.Integer(0))

# Create var bind list for GetNext
next_var_bind_list = rfc1157.VarBindList()
next_var_bind = rfc1157.VarBind()
next_var_bind.setComponentByName('name', univ.ObjectIdentifier('1.3.6.1.2.1.1'))  # system branch

# Empty value for GetNext
next_obj_syntax = rfc1155.ObjectSyntax()
next_simple_syntax = rfc1155.SimpleSyntax()
next_simple_syntax.setComponentByName('empty', univ.Null())
next_obj_syntax.setComponentByName('simple', next_simple_syntax)
next_var_bind.setComponentByName('value', next_obj_syntax)

next_var_bind_list.setComponentByPosition(0, next_var_bind)
get_next_pdu.setComponentByName('variable-bindings', next_var_bind_list)

print(f"\nGetNext Request PDU type: {type(get_next_pdu)}")
print(f"GetNext Request ID: {get_next_pdu.getComponentByName('request-id')}")

# Encode SNMP message
encoded_message = encoder.encode(snmp_message)
print(f"\nSNMP message encoded size: {len(encoded_message)} bytes")

# Available SNMP v1 structures
print(f"\nSNMPv1 PDU types available:")
pdu_types = [attr for attr in dir(rfc1157) if 'PDU' in attr or 'Request' in attr]
for pdu in pdu_types[:5]:
    print(f"  {pdu}")

print("Example 4 completed successfully!")

In [ ]:
# Example 5: Available Protocol Modules and Standards Coverage
import pyasn1_modules
import inspect

print("=== Protocol Modules and Standards Coverage ===")

# Discover available RFC modules
rfc_modules = [attr for attr in dir(pyasn1_modules) if attr.startswith('rfc')]
print(f"Available RFC modules: {len(rfc_modules)}")
print(f"First 10 RFC modules: {rfc_modules[:10]}")

# Discover PKCS modules  
pkcs_modules = [attr for attr in dir(pyasn1_modules) if 'pkcs' in attr.lower()]
print(f"PKCS modules: {pkcs_modules}")

# Check some common protocols
common_protocols = ['rfc5280', 'rfc3447', 'rfc5652', 'rfc3416', 'rfc2459']
for protocol in common_protocols:
    if hasattr(pyasn1_modules, protocol):
        module = getattr(pyasn1_modules, protocol)
        structures = [attr for attr in dir(module) if not attr.startswith('_') and inspect.isclass(getattr(module, attr, None))]
        print(f"{protocol}: {len(structures)} ASN.1 structures")

# Test module imports and basic functionality
test_modules = ['rfc5280', 'rfc3447', 'rfc5652']
for module_name in test_modules:
    if hasattr(pyasn1_modules, module_name):
        module = getattr(pyasn1_modules, module_name)
        try:
            # Try to get first class from module
            classes = [attr for attr in dir(module) if inspect.isclass(getattr(module, attr, None))]
            if classes:
                first_class = getattr(module, classes[0])
                instance = first_class()
                print(f"{module_name}.{classes[0]}: instantiated successfully")
        except Exception as e:
            print(f"{module_name}: Error in basic test - {e}")
            raise

# Summary statistics
total_modules = len([attr for attr in dir(pyasn1_modules) if not attr.startswith('_')])
print(f"\nTotal protocol modules available: {total_modules}")
print(f"RFC-based modules: {len(rfc_modules)}")
print(f"Coverage: X.509, PKCS, CMS, SNMP, and many other standards")

print("Example 5 completed successfully!")

In [ ]:
# Additional: Cross-Protocol Integration and Real-World Usage
from pyasn1_modules import rfc5280, rfc3447, rfc5652

print("=== Cross-Protocol Integration Example ===")

# Create a SubjectPublicKeyInfo using PKCS#1 RSA key within X.509 structure
subject_public_key_info = rfc5280.SubjectPublicKeyInfo()

# Set algorithm identifier for RSA encryption
algorithm_identifier = rfc5280.AlgorithmIdentifier()
algorithm_identifier.setComponentByName('algorithm', univ.ObjectIdentifier('1.2.840.113549.1.1.1'))  # rsaEncryption
subject_public_key_info.setComponentByName('algorithm', algorithm_identifier)

# Create RSA public key and encode it
rsa_public_key = rfc3447.RSAPublicKey()
rsa_public_key.setComponentByName('modulus', univ.Integer(0x1234567890abcdef1234567890abcdef))
rsa_public_key.setComponentByName('publicExponent', univ.Integer(65537))

encoded_rsa_key = encoder.encode(rsa_public_key)
subject_public_key_info.setComponentByName('subjectPublicKey', univ.BitString.fromHexString(encoded_rsa_key.hex()))

print(f"Integrated SubjectPublicKeyInfo created: {type(subject_public_key_info)}")
print(f"Algorithm OID: {algorithm_identifier.getComponentByName('algorithm')}")

# Demonstrate certificate extension usage
extensions = rfc5280.Extensions()
extension = rfc5280.Extension()
extension.setComponentByName('extnID', univ.ObjectIdentifier('2.5.29.15'))  # keyUsage
extension.setComponentByName('critical', univ.Boolean(True))
extension.setComponentByName('extnValue', univ.OctetString(b'\x03\x02\x01\x86'))  # Digital Signature, Key Encipherment

extensions.setComponentByPosition(0, extension)
print(f"Certificate extension created: {extension.getComponentByName('extnID')}")
print(f"Extension critical: {extension.getComponentByName('critical')}")

# Show interoperability between modules
print(f"X.509 structures integrate with PKCS#1: RSA keys fit in certificates")
print(f"CMS can reference X.509 certificates: ContentInfo → SignedData → Certificate")

# Encode final structure
encoded_subject_key_info = encoder.encode(subject_public_key_info)
print(f"Final integrated structure size: {len(encoded_subject_key_info)} bytes")
print("Cross-protocol integration completed successfully!")